# import libraries

In [35]:
import pandas as pd
import numpy as np
import joblib
import json

from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# load data

In [36]:
data_path = Path("../data/processed/cleaned_salaries.csv")

df = pd.read_csv(data_path)

df.head()

,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,work_setting,company_location,company_size,salary_in_usd
0,2020,Mid-level,Full-time,Data Scientist,DE,0,On-site,DE,Large,79833
1,2020,Senior-level,Full-time,Machine Learning Scientist,JP,0,On-site,JP,Small,260000
2,2020,Senior-level,Full-time,Big Data Engineer,GB,50,Hybrid,GB,Medium,109024
3,2020,Mid-level,Full-time,Product Data Analyst,HN,0,On-site,HN,Small,20000
4,2020,Senior-level,Full-time,Machine Learning Engineer,US,50,Hybrid,US,Large,150000


In [37]:
df. info()

<class 'pandas.DataFrame'>
RangeIndex: 565 entries, 0 to 564
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   work_year           565 non-null    int64
 1   experience_level    565 non-null    str  
 2   employment_type     565 non-null    str  
 3   job_title           565 non-null    str  
 4   employee_residence  565 non-null    str  
 5   remote_ratio        565 non-null    int64
 6   work_setting        565 non-null    str  
 7   company_location    565 non-null    str  
 8   company_size        565 non-null    str  
 9   salary_in_usd       565 non-null    int64
dtypes: int64(3), str(7)
memory usage: 44.3 KB


Choose features and target

Use salary_in_usd as the target.

Do not use salary, salary_currency, or work_setting for training.

work_setting is only a readable version of remote_ratio, so we should avoid using both.

In [38]:
feature_columns = [
    "work_year",
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "remote_ratio",
    "company_location",
    "company_size",
]

target_column = "salary_in_usd"

X = df[feature_columns]
y = df[target_column]

# split data

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [40]:
X_train.shape

(452, 8)

In [41]:
X_test.shape

(113, 8)

# define numeric and categorical columns

In [42]:
numeric_features = ["work_year", "remote_ratio"]

categorical_features = [
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "company_location",
    "company_size",
]

# create preprocessing pipeline

In [43]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)

# create decision tree pipeline

In [44]:
model = DecisionTreeRegressor(random_state=42)

pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

# train a basic model

In [45]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['work_year','experience_level','employment_type',...,'remote_ratio', 'company_location','company_size']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatic

# evaluate the basic model

In [46]:
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Basic Decision Tree Results")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²:   {r2:.4f}")

Basic Decision Tree Results
MAE:  32,912.47
RMSE: 54,894.05
R²:   0.3727


# tune the decision tree model

In [47]:
param_grid = {
    "model__max_depth": [3, 5, 7, 10, 15, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": [None, "sqrt", "log2"],
}

In [48]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 288 candidates, totalling 1440 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [3, 5, ...], 'model__max_features': [None, 'sqrt', ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about 

# show best parameters

In [49]:
grid_search.best_params_

{'model__max_depth': 7,
 'model__max_features': None,
 'model__min_samples_leaf': 2,
 'model__min_samples_split': 5}

# evalute tuned model

In [50]:
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)

mae_best = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best = r2_score(y_test, y_pred_best)

print("Tuned Decision Tree Results")
print(f"MAE:  {mae_best:,.2f}")
print(f"RMSE: {rmse_best:,.2f}")
print(f"R²:   {r2_best:.4f}")

Tuned Decision Tree Results
MAE:  32,331.41
RMSE: 52,882.32
R²:   0.4179


# Compare against a simple baseline

This checks whether your model is actually useful.

The baseline always predicts the median salary from the training data.

In [51]:
baseline_prediction = np.full(shape=len(y_test), fill_value=y_train.median())

baseline_mae = mean_absolute_error(y_test, baseline_prediction)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_prediction))
baseline_r2 = r2_score(y_test, baseline_prediction)

print("Baseline Results")
print(f"MAE:  {baseline_mae:,.2f}")
print(f"RMSE: {baseline_rmse:,.2f}")
print(f"R²:   {baseline_r2:.4f}")

Baseline Results
MAE:  50,958.96
RMSE: 69,822.65
R²:   -0.0149


## sample input

In [52]:
sample_input = pd.DataFrame(
    [
        {
            "work_year": 2024,
            "experience_level": "Senior-level",
            "employment_type": "Full-time",
            "job_title": "Data Scientist",
            "employee_residence": "US",
            "remote_ratio": 100,
            "company_location": "US",
            "company_size": "Medium",
        }
    ]
)

sample_prediction = best_model.predict(sample_input)[0]

print(f"Predicted salary: ${sample_prediction:,.2f}")

Predicted salary: $160,832.67


# save model

In [ ]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "salary_decision_tree_pipeline.joblib"

joblib.dump(best_model, model_path)

print(f"Model saved to: {model_path}")

# save model metrics

In [ ]:
metrics = {
    "model_name": "DecisionTreeRegressor",
    "target": target_column,
    "features": feature_columns,
    "best_params": grid_search.best_params_,
    "mae": mae_best,
    "rmse": rmse_best,
    "r2": r2_best,
    "baseline_mae": baseline_mae,
    "baseline_rmse": baseline_rmse,
    "baseline_r2": baseline_r2,
}

metrics_path = models_dir / "model_metrics.json"

with open(metrics_path, "w") as file:
    json.dump(metrics, file, indent=4)

print(f"Metrics saved to: {metrics_path}")